# Experimento MAREA — Variante: **MAREA-2** (microcolonias)

**MAREA-2 — metaheuristica bacteriana por microcolonias (ciclo de vida de Cobetia sp.)**

Rediseño a partir de lo que dejo el benchmark OE1:

- MAREA_REL converge pero **no escapa**: de 50k a 500k FES el error en unimodales
  baja a 0.02-0.05x, en multimodales solo a 0.89-0.92x (ABC: 0.46-0.71x).
- Pierde contra ABC exactamente en multimodales y composicion (D50/500k: 6/1/17 y 6/0/15).
- Causa: la ecuacion (1) de AGEO tiene **un solo atractor** (X_g, mu globales).
- MAREA-bio fallo porque B~ satura en la iteracion 2, el Levy salia desde X_g y
  la dispersion era probabilistica y no por estado de la busqueda.

### Componentes (un flag cada uno, ver `MAREA2_especificacion.md`)

| flag | mecanismo | analogo biologico |
|---|---|---|
| `colonies` | k-means dinamico; X_c*, mu_c, sigma_c por colonia | microcolonias del biofilm |
| `starvation` | tau_c iteraciones sin mejora > limit -> dispersa la mitad peor (Levy desde su propia posicion), conserva la mejor | desprendimiento por inanicion; celulas persistentes |
| `quorum` | regla 1/5 por colonia sobre gamma: exito alto = paso mayor (planctonico), bajo = paso menor (sesil) | trade-off mu_max/biofilm del paper (ec. 7) |
| `pop_schedule='cobetia'` | capacidad de carga N(u) con la logistica en log-tiempo **ajustada en el paper** (Tabla 1: xmid=3.429, s=0.784) | consumo de sustrato en cultivo batch |
| `lag` | primeros 5% de FES: alpha_min, beta_max, N0 | fase lag |
| `nutrient_memory` | archivo de cuencas agotadas; re-siembra si cae cerca | sustrato agotado |
| `noise='diff'` | perturbacion = diferencia de dos vecinas + 0.3 sigma_c t_nu, **Student-t nu=3.05** (familia TF del paper, Tabla 1) | transferencia horizontal; colas pesadas medidas en Cobetia |
| `greedy` | aceptar solo si mejora | seleccion |

Con todo apagado (`config('AGEO_REL')`) el algoritmo **es MAREA_REL**: sirve de control.

Nota sobre la cadena paper -> AGEO -> MAREA-2: la ecuacion (1) de AGEO no toma nada
especifico del paper de Frontiers (es un esqueleto de enjambre generico). Las tres piezas
que hacen literal el vinculo con Cobetia son la logistica ajustada (C4), el trade-off
mu_max/biofilm como ley de paso (C3) y la Student-t nu=3.05 (C7); las tres tienen su brazo
de ablacion (`gompertz`/`lineal`, `sin_quorum`, `ruido_gauss`). Ver especificacion, sec. 8.

### Protocolo
- 71 funciones CEC 2014 (30) + CEC 2017 (29) + CEC 2022 (12)
- 10D y 50D (CEC2022: 10D y 20D) | 51 corridas | guardado incremental
- **Una sola** configuracion de parametros para las 71 funciones y los 4 MaxFES.
  Se fija en la Fase A (ablacion, 15 corridas) y **no se toca** despues.
- El presupuesto se controla en **FES**, no en iteraciones: con reduccion de
  poblacion `iters > T`. El smoke test comprueba `fes <= MaxFES`, no `iters == T`.

### Orden de ejecucion
1. Celdas 1-2 (entorno, smoke test).
2. **Fase A** — ablacion barata (seccion 3). Decide que componentes sobreviven.
3. Fijar `CONFIG` con lo que sobrevivio y correr el protocolo (secciones 4-7).

> Requiere `marea2.py` y `ablacion_marea2.py` en el directorio del proyecto.


In [ ]:
import sys, os
import numpy as np
import warnings
warnings.filterwarnings('ignore')

PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from experimento_config import (
    run_experiment, load_partial, CEC_REGISTRY,
    MAXFES, DIMS, N_RUNS, POP_SIZE, RESULTS_DIR
)

CONFIG       = 'FULL'   # cambiar solo tras la Fase A (p.ej. 'sin_memoria' si la memoria no aporta)
# El nombre lleva la variante: run_experiment IGNORA alg_fn y reimporta por
# nombre con experimento_config._import_alg. Si CONFIG no viaja en ALG_NAME,
# el protocolo corre FULL aunque hayas cambiado CONFIG.
ALG_NAME     = 'MAREA2' if CONFIG == 'FULL' else f'MAREA2_{CONFIG}'
N_WORKERS    = 8        # el Mac mini tiene 24 cores; dejar aire si hay otra corrida
RUN_ABLACION = False    # Fase A: 14 configs x 2 dims x {5k, 50k} x 15 corridas (~una tarde)
RUN_A2       = False    # Fase A2: cadena desde sin_colonias (ya corrida el 2026-09-10)
RUN_A3       = True     # Fase A3: 2x2 colonias x perturbacion a 500k (~12 h con 8 workers)
RUN_PROTOCOLO = False   # secciones 4-7: solo con CONFIG ya fijado tras la Fase A/A2/A3
RUN_HEAVY    = False    # 50k / 500k / 5M del protocolo completo solo con True

from marea2 import marea2, config, MAREA2Params, ABLATION_CONFIGS, A2_CONFIGS
p = config(CONFIG)

def alg_fn(f, bounds, N, T, seed):
    p.N0 = N
    return marea2(f, bounds, p, maxfes=N * T, seed=seed)

print(f'\u2713 {ALG_NAME} cargado correctamente  (config = {CONFIG})')
print(f'  colonias={p.colonies} inanicion={p.starvation} quorum={p.quorum} '
      f'schedule={p.pop_schedule} lag={p.lag} memoria={p.nutrient_memory} '
      f'ruido={p.noise} greedy={p.greedy}')
print(f'  alpha=[{p.alpha_min},{p.alpha_max}] beta_max={p.beta_max} gamma_max={p.gamma_max} '
      f'N_min={p.N_min} celulas/colonia={p.cells_per_colony}')
print(f'  schedule cobetia: xmid={p.cob_xmid} s={p.cob_scal} tmax={p.cob_tmax}h | '
      f'ruido por eje: {p.axis_dist} (nu={p.student_nu})')
print(f'  Resultados en: {RESULTS_DIR}/')
print(f'  Funciones CEC disponibles: {len(CEC_REGISTRY)}')
print(f'  Workers: {N_WORKERS} | RUN_ABLACION: {RUN_ABLACION} | RUN_A2: {RUN_A2} | RUN_A3: {RUN_A3} | '
      f'RUN_PROTOCOLO: {RUN_PROTOCOLO} | RUN_HEAVY: {RUN_HEAVY}')


## 1. Verificacion de entorno — **no saltarse**

Solo si dice **ESTE ES EL ENTORNO CORRECTO** los resultados seran comparables
con el resto del benchmark. Se lanza con `sys.executable`, el interprete real
del kernel: cambiar el kernel en el menu no reinicia el proceso.


In [ ]:
!{sys.executable} verificar_entorno.py


## 2. Smoke test + auditoria de presupuesto

1. **Agota MaxFES** — `fes` debe quedar en `[MaxFES - POP_SIZE, MaxFES]`.
   (No se comprueba `iters == T`: con la logistica de Cobetia la poblacion se reduce y hay mas iteraciones.)
2. **No se pasa** — el contador externo debe dar exactamente `<= 1.00x`. MAREA-2
   contabiliza cada evaluacion, incluidas las de las celulas dispersadas.
3. **La config elegida realmente difiere del control** `AGEO_REL` (= MAREA_REL).
4. **Diagnostico** — cuantas colonias hay al final y cuantas dispersiones hubo,
   para confirmar que los mecanismos se activan en CEC y no solo en el smoke set.


In [ ]:
from experimento_config import get_problems_for_dim
import time

problems_10d = get_problems_for_dim(10)
test_prob    = problems_10d[0]
print(f'Probando con: {test_prob["name"]} | dim={test_prob["dim"]}')

T_TEST = 62   # 62 x 80 = 4,960 FES ~ MaxFES=5,000
maxfes = POP_SIZE * T_TEST

t0 = time.time()
for seed in range(3):
    r = alg_fn(test_prob['f'], test_prob['bounds'], N=POP_SIZE, T=T_TEST, seed=seed)
    print(f'  seed={seed}  best_f={r["best_f"]:.4e}  fes={r["fes"]}/{maxfes}  iters={r["iters"]}')
    assert r['fes'] <= maxfes,            'se paso del presupuesto'
    assert r['fes'] >= maxfes - POP_SIZE, 'paro antes de agotar el presupuesto'
print(f'Tiempo: {time.time()-t0:.2f}s')

class _Contador:
    def __init__(self, f): self.f, self.n = f, 0
    def __call__(self, x):
        self.n += 1
        return self.f(x)

cnt = _Contador(test_prob['f'])
alg_fn(cnt, test_prob['bounds'], N=POP_SIZE, T=T_TEST, seed=0)
print(f'FES reales = {cnt.n:,}   nominal = {maxfes:,}   ratio = {cnt.n/maxfes:.2f}x')
assert cnt.n <= maxfes, 'el contador externo detecta mas evaluaciones que MaxFES'

# la config elegida difiere del control?
q = config('AGEO_REL'); q.N0 = POP_SIZE
ctrl = marea2(test_prob['f'], test_prob['bounds'], q, maxfes=maxfes, seed=0)['best_f']
this = alg_fn(test_prob['f'], test_prob['bounds'], N=POP_SIZE, T=T_TEST, seed=0)['best_f']
print(f'AGEO_REL (control) = {ctrl:.4e}   {CONFIG} = {this:.4e}')
assert ctrl != this, f'{CONFIG} da lo mismo que el control: revisa los flags'

# diagnostico: se activan los mecanismos con presupuesto realista?
pd_ = config(CONFIG); pd_.N0 = POP_SIZE; pd_.record_history = True
rh = marea2(test_prob['f'], test_prob['bounds'], pd_, maxfes=50000, seed=0)
h = np.array(rh['history'])            # columnas: fes, gbest, k colonias, N
print(f'50k FES: colonias inicio/fin = {int(h[0,2])}/{int(h[-1,2])}   '
      f'N inicio/fin = {int(h[0,3])}/{int(h[-1,3])}   best_f = {rh["best_f"]:.4e}')
print('\u2713 Listo para el experimento.')


## 3. Fase A — ablacion (screening barato) ← **ejecutar antes del protocolo**

14 configuraciones x D10/D50 x {5k, 50k} x 15 corridas. Guarda en
`RESULTS_DIR/ablacion/MAREA2_<config>_D<d>_FES<mf>.json`, formato identico al del
resto del benchmark; se puede interrumpir y reanudar.

Criterio de aceptacion de un componente C (especificacion, sec. 6): FULL vs `sin_C` debe
tener **B > W en la clase que C ataca** y **W <= B + 2 en unimodales**. Si no lo cumple en
ninguna celda, C se elimina y `CONFIG` pasa a `sin_C`.

Tres brazos contrastan la fidelidad al paper: `gompertz` y `lineal` (LPSR) contra la logistica
de Cobetia; `ruido_gauss` contra la Student-t nu=3.05.

Requiere `RUN_ABLACION = True`. Coste aproximado: 5k -> ~25 min, 50k -> ~3.5 h con 8 workers.


In [ ]:
import ablacion_marea2 as abl

if RUN_ABLACION:
    for mf in (5000, 50000):
        abl.run(ABLATION_CONFIGS, dims=list(DIMS), maxfes=mf, runs=15, workers=N_WORKERS)
else:
    print('Saltada (RUN_ABLACION = False). Coste: ~20 min (5k) + ~3 h (50k) con 8 workers')


### Fase A2 — cadena de un cambio a la vez desde `sin_colonias`

La Fase A mostro que las colonias perjudican (sobre todo en D50), que memoria y lag son inertes,
que la perturbacion por eje gana a la diferencia de vecinas en D50 y que la inanicion no se
activa en 5k FES con `limit = 10 + 2D`. La referencia pasa a ser `sin_colonias` (ya corrida) y
cada brazo cambia **una** cosa respecto a ella:

| brazo | cambio respecto a `sin_colonias` |
|---|---|
| `A2_eje_student` | perturbacion por eje Student-t, sin diferencia de vecinas (= `ruido_normal`: **no** es gaussiana) |
| `A2_eje_gauss` | perturbacion por eje gaussiana |
| `A2_limit` | inanicion con `limit_mode='budget'` (limit 12 en 5k) |
| `A2_lite` | sin memoria ni lag (los dos inertes) |
| `A2_gompertz` | Gompertz en vez de la logistica de Cobetia |
| `A2_todo` | todos los cambios juntos (conserva Student-t) |

`A2_limit` solo se corre a 5k: a 50k su limit es el mismo que `'dim'` y da resultados identicos
a `sin_colonias`. En el reporte, las `A2_*` se comparan contra `sin_colonias`, no contra FULL.

Si en `ablacion/` hay JSON de la A2 anterior (`A2_sincol_*`, `A2_lite_normal`, `A2_lite_gompertz`),
la celda aparta esos archivos y el viejo `A2_lite` (otros parametros bajo el mismo nombre) a
`ablacion/obsoletos_A2_v1/`, y reutiliza `A2_sincol_normal` como `A2_eje_student` (mismos parametros).

Requiere `RUN_A2 = True`. Coste aproximado: 5k -> ~10 min, 50k -> ~1-1.5 h con 8 workers.
Se puede interrumpir y reanudar.

In [ ]:
import glob, shutil

# JSON de la A2 anterior: apartarlos para que no se tomen como corridas hechas
viejos = [f for c in ('A2_sincol', 'A2_sincol_normal', 'A2_sincol_limit', 'A2_lite_normal', 'A2_lite_gompertz')
          for f in glob.glob(os.path.join(abl.ABL_DIR, f'MAREA2_{c}_D*_FES*.json'))]
if viejos:
    obsoletos = os.path.join(abl.ABL_DIR, 'obsoletos_A2_v1')
    os.makedirs(obsoletos, exist_ok=True)
    for f in glob.glob(os.path.join(abl.ABL_DIR, 'MAREA2_A2_sincol_normal_D*_FES*.json')):
        dst = f.replace('A2_sincol_normal', 'A2_eje_student')
        if not os.path.exists(dst):
            shutil.copy2(f, dst); print(f'  reutilizado: {os.path.basename(dst)}')
    for f in viejos + glob.glob(os.path.join(abl.ABL_DIR, 'MAREA2_A2_lite_D*_FES*.json')):
        shutil.move(f, os.path.join(obsoletos, os.path.basename(f)))
        print(f'  apartado   : {os.path.basename(f)}')

A2_5K  = [c for c in A2_CONFIGS if c != 'sin_colonias']   # sin_colonias ya esta de la Fase A
A2_50K = [c for c in A2_5K if c != 'A2_limit']            # a 50k A2_limit == sin_colonias

if RUN_A2:
    abl.run(A2_5K,  dims=list(DIMS), maxfes=5000,  runs=15, workers=N_WORKERS)
    abl.run(A2_50K, dims=list(DIMS), maxfes=50000, runs=15, workers=N_WORKERS)
else:
    print('Saltada (RUN_A2 = False). Coste: ~10 min (5k) + ~1-1.5 h (50k) con 8 workers')

### Reporte de la ablacion

Por celda: rank medio de las variantes de la Fase A y de la A2 junto con los competidores que
tengan JSON (se usan sus primeras 15 semillas), B/E/W **por clase de funcion** de la referencia
contra cada variante (columna `ref`: `FULL` para la Fase A, `sin_colonias` para las `A2_*`),
y B/E/W de FULL contra los competidores. `A2_limit` no aparece a 50k: no se corre ahi.


In [ ]:
for mf in (5000, 50000):
    abl.report(dims=list(DIMS), maxfes=mf, runs=15)


### Metrica de escape — aprovechamiento del presupuesto

Mediana de `err(50k) / err(5k)` por clase. Es la metrica que delato a MAREA_REL
(0.89-0.92 en multimodales de 50k a 500k). Un componente que ataca lo multimodal debe
bajar este ratio respecto a su `sin_X`.


In [ ]:
abl.ratio(dims=list(DIMS), fes_a=5000, fes_b=50000, runs=15)


### Fase A3 — screening a 500k: colonias x tipo de perturbacion

La A2 no alcanzo para elegir `CONFIG`: el efecto de las colonias y de la perturbacion por eje
depende de la dimension y del presupuesto, y el screening a 5k/50k no predice 500k. Resultado
de la A2 (B/E/W de la variante contra `sin_colonias`):

| | D10 5k | D50 5k | D10 50k | D50 50k |
|---|---|---|---|---|
| `A2_eje_student` (por eje) | 18/36/17 | 46/19/6 | 3/34/34 | 41/13/17 |
| `FULL` (con colonias) | 1/48/22 | 1/24/46 | 8/47/16 | 0/18/53 |

y sin embargo el protocolo de `FULL` (51 corridas) es **1º del benchmark a 500k** en D10 y D50.
Lo que si quedo decidido en la A2: Student-t >= gaussiana, logistica de Cobetia >= Gompertz;
memoria, lag y `limit_mode='budget'` inertes.

Cruce 2x2 a 500k, 15 corridas, D10 y D50:

| | diferencia de vecinas | por eje (Student-t) |
|---|---|---|
| **con colonias** | `FULL` | `ruido_normal` |
| **sin colonias** | `sin_colonias` | `A2_eje_student` |

`FULL` se recorre aqui: el protocolo usa otras semillas que la ablacion (0/1065 coinciden),
asi que sus 51 corridas no se pueden parear con estas.

Requiere `RUN_A3 = True`. Coste aproximado: ~12 h con 8 workers, ~5 h con 20 (subir `N_WORKERS`
en la celda de carga si el Mac mini no tiene otra corrida). Corre primero D10 (las 4 configs)
y despues D50; se puede interrumpir y reanudar.

In [ ]:
A3_CONFIGS = ['FULL', 'sin_colonias', 'ruido_normal', 'A2_eje_student']

if RUN_A3:
    abl.run(A3_CONFIGS, dims=list(DIMS), maxfes=500000, runs=15, workers=N_WORKERS)
else:
    print('Saltada (RUN_A3 = False). Coste: ~12 h con 8 workers, ~5 h con 20')

### Reporte de la Fase A3

1. `abl.report` a 500k: rank medio de las 4 configs junto con los competidores (primeras 15
   semillas de sus 51) y la tabla de ablacion (B/E/W **de la referencia**: `FULL`, o
   `sin_colonias` para `A2_eje_student`).
2. Efectos del 2x2, B/E/W **del primero** (B = el primero gana): el efecto de las colonias con
   cada perturbacion, y el de la perturbacion por eje con y sin colonias.
3. Metrica de escape `err(500k) / err(50k)` por clase.

Criterio: se queda la celda del 2x2 que gane a 500k en las dos dimensiones. Si el ganador
depende de la dimension, la hipotesis a probar es la regla "perturbacion por eje cuando la
colonia tiene <= D celulas" (`N_min = 20` < D+1 en D50: las diferencias de vecinas solo
cubren un subespacio de 19 dimensiones).

In [ ]:
from experimento_config import get_problems_for_dim

abl.report(dims=list(DIMS), maxfes=500000, runs=15, cfgs=A3_CONFIGS)

EFECTOS = [('FULL',           'sin_colonias',   'colonias | dif. vecinas'),
           ('ruido_normal',   'A2_eje_student', 'colonias | por eje'),
           ('ruido_normal',   'FULL',           'por eje | con colonias'),
           ('A2_eje_student', 'sin_colonias',   'por eje | sin colonias')]
for dim in DIMS:
    probs = get_problems_for_dim(dim); n = len(probs)
    cls = [abl.clase(pr['name'], i) for i, pr in enumerate(probs)]
    M = {c: abl.load(c, dim, 500000, 15, n) for c in A3_CONFIGS}
    print(f'\nD={dim}  MaxFES=500,000  efectos del 2x2  (B/E/W del primero: B = el primero gana)')
    print(f'  {"efecto":26s}' + ''.join(f'{c:>14s}' for c in abl.CLASES) + f'{"total":>12s}')
    for a, b, lab in EFECTOS:
        if M[a] is None or M[b] is None:
            print(f'  {lab:26s}  (faltan corridas de {a if M[a] is None else b})'); continue
        row = [abl._bew(M[a], M[b], [q for q in range(n) if cls[q] == c]) for c in abl.CLASES]
        print(f'  {lab:26s}' + ''.join(f'{r:>14s}' for r in row) + f'{abl._bew(M[a], M[b], range(n)):>12s}')

abl.ratio(dims=list(DIMS), fes_a=50000, fes_b=500000, runs=15, cfgs=A3_CONFIGS)

## 4. Experimento — MaxFES = 5,000
*Caso de uso: optimizacion costosa / tiempo-real*

Coste: ~6 min con 8 workers. Escribe en `MAREA2_D{10,50}_FES5000.json` — no pisa nada.
Solo tiene sentido con `CONFIG` ya fijado tras la Fase A/A2/A3. Requiere `RUN_PROTOCOLO = True`.


In [ ]:
# ── MaxFES = 5,000 ──────────────────────────────
if RUN_PROTOCOLO:
    for ndim in DIMS:
        run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=5000, n_workers=N_WORKERS)
else:
    print('Saltada (RUN_PROTOCOLO = False): fijar CONFIG tras la Fase A/A2/A3. Coste: ~6 min con 8 workers')


## 5. Experimento — MaxFES = 50,000

Coste: ~20 min (D=10) + ~45 min (D=50) con 8 workers. Requiere `RUN_HEAVY = True`.


In [ ]:
if RUN_PROTOCOLO and RUN_HEAVY:
    for ndim in DIMS:
        run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=50000, n_workers=N_WORKERS)
else:
    print('Saltada (RUN_PROTOCOLO/RUN_HEAVY = False). Coste: ~20 min (D=10) + ~45 min (D=50) con 8 workers')


## 6. Experimento — MaxFES = 500,000

Coste: ~3 h (D=10) + ~7 h (D=50) con 8 workers. Requiere `RUN_HEAVY = True`.


In [ ]:
if RUN_PROTOCOLO and RUN_HEAVY:
    for ndim in DIMS:
        run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=500000, n_workers=N_WORKERS)
else:
    print('Saltada (RUN_PROTOCOLO/RUN_HEAVY = False). Coste: ~3 h (D=10) + ~7 h (D=50) con 8 workers')


## 7. Experimento — MaxFES = 5,000,000

Coste: ~10x el de 500k — noche completa de maquina. **Bloqueada:** el diseño experimental cierra en 500k; el 5M se descartó. Solo corre con
`RUN_5M = True` (dentro de la celda) además de `RUN_PROTOCOLO` y `RUN_HEAVY`.


In [ ]:
RUN_5M = False   # 5M descartado: el diseño experimental cierra en 500k
if RUN_5M and RUN_PROTOCOLO and RUN_HEAVY:
    for ndim in DIMS:
        run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=5000000, n_workers=N_WORKERS)
else:
    print('Bloqueada: 5M descartado (el diseño cierra en 500k). Ver RUN_5M')


## Resumen de resultados


In [ ]:
print(f'\n=== RESUMEN {ALG_NAME} ({CONFIG}) ===')
print(f'{"Config":<30} {"Completadas":>12} {"Mean":>14} {"Best":>14}')
print('-' * 72)
for mf in MAXFES:
    for nd in DIMS:
        vals = list(load_partial(ALG_NAME, nd, mf).values())
        label = f'D={nd:2d} MaxFES={mf:>10,}'
        if vals:
            print(f'{label:<30} {len(vals):>12} {np.mean(vals):>14.4e} {np.min(vals):>14.4e}')
        else:
            print(f'{label:<30}   (sin resultados aun)')


## Comparacion — MAREA-2 contra las variantes MAREA y los competidores

Metrica: media de las 51 corridas, empates a 1e-8 comparten rango (setup, sec. 5 y 6).
Wilcoxon de rangos con signo pareado por semilla (setup, sec. 7).

Tres bloques por celda:
- **rank medio** en el pool completo (FA y CLONALG ya con presupuesto corregido).
- **MAREA2 vs MAREA_REL** por clase de funcion — la pregunta del rediseño: ¿escapa donde
  MAREA_REL se quedaba atrapada (multimodal, composicion) sin perder en unimodales?
- **MAREA2 vs cada competidor** por clase.


In [ ]:
import json, re
from scipy.stats import rankdata, wilcoxon

ARMAS        = [ALG_NAME, 'MAREA_REL', 'MAREABIO_REL', 'MAREA', 'MAREABIO']
COMPETIDORES = ['ABC', 'DE', 'FA', 'GWO', 'PSO', 'WOA', 'CLONALG']
CLASES       = ['unimodal', 'multimodal', 'hibrida', 'composicion']

def _load(alg, nd, mf):
    for d in (RESULTS_DIR, PROJECT_DIR, '.'):
        path = os.path.join(d, f'{alg}_D{nd}_FES{mf}.json')
        if os.path.exists(path):
            M = np.full((71, N_RUNS), np.nan)
            for k, v in json.load(open(path)).items():
                a, b = k.split('_')
                M[int(a[3:]), int(b[3:])] = v
            return M if np.isfinite(M).mean() > 0.99 else None
    return None

def _clase(nombre, idx):
    m = re.search(r'F(\d+).*?(2014|2017|2022)', nombre)
    if m: n, y = int(m.group(1)), m.group(2)
    else:
        if idx < 30:   n, y = idx + 1, '2014'
        elif idx < 59: n, y = idx - 29 + (1 if idx > 30 else 0), '2017'
        else:          n, y = idx - 58, '2022'
    if y == '2014': return 'unimodal' if n <= 3 else 'multimodal' if n <= 16 else 'hibrida' if n <= 22 else 'composicion'
    if y == '2017': return 'unimodal' if n <= 3 else 'multimodal' if n <= 10 else 'hibrida' if n <= 20 else 'composicion'
    return 'unimodal' if n == 1 else 'multimodal' if n <= 5 else 'hibrida' if n <= 8 else 'composicion'

def _ranks(col, tol=1e-8):
    r = rankdata(col); o = np.argsort(col); out = r.copy(); i = 0
    while i < len(o):
        j = i
        while j + 1 < len(o) and abs(col[o[j+1]] - col[o[i]]) < tol:
            j += 1
        if j > i:
            a = np.mean([r[o[k]] for k in range(i, j+1)])
            for k in range(i, j+1):
                out[o[k]] = a
        i = j + 1
    return out

def _bew(X, Y, sel):
    mx, my = X.mean(1), Y.mean(1)
    B = E = W = 0
    for q in sel:
        try:    pv = wilcoxon(X[q], Y[q])[1]
        except Exception: pv = 1.0
        if pv >= 0.05:      E += 1
        elif mx[q] < my[q]: B += 1
        else:               W += 1
    return f'{B}/{E}/{W}'

for mf in MAXFES:
    for nd in DIMS:
        me = _load(ALG_NAME, nd, mf)
        if me is None:
            continue
        probs = get_problems_for_dim(nd)
        cls = [_clase(pr['name'], i) for i, pr in enumerate(probs)]
        pool = {ALG_NAME: me}
        for a in ARMAS[1:] + COMPETIDORES:
            m = _load(a, nd, mf)
            if m is not None:
                pool[a] = m
        print(f'\n{"="*74}\nD={nd}   MaxFES={mf:,}   algoritmos: {len(pool)}')

        names = list(pool)
        means = np.array([pool[a].mean(1) for a in names])
        R  = np.column_stack([_ranks(means[:, q]) for q in range(71)])
        mr = R.mean(1); o = np.argsort(mr)
        pos = [names[i] for i in o].index(ALG_NAME) + 1
        print(f'  rank medio  : {ALG_NAME} {pos}o de {len(names)}')
        print('      ' + '  '.join(f'{names[i]}:{mr[i]:.2f}' for i in o))

        print(f'\n  {"MAREA2 vs":14s}' + ''.join(f'{c:>14s}' for c in CLASES) + f'{"total":>12s}')
        for a in ARMAS[1:] + COMPETIDORES:
            if a not in pool:
                continue
            row = [_bew(me, pool[a], [q for q in range(71) if cls[q] == c]) for c in CLASES]
            print(f'  {a:14s}' + ''.join(f'{r:>14s}' for r in row) + f'{_bew(me, pool[a], range(71)):>12s}')


## Metrica de escape en el protocolo completo

Mediana de `err(500k) / err(50k)` por clase. MAREA_REL daba 0.89-0.92 en multimodales,
ABC 0.46-0.71. Si MAREA-2 no baja de ahi, los mecanismos de dispersion no estan
funcionando en CEC aunque hayan pasado la ablacion.


In [ ]:
for nd in DIMS:
    probs = get_problems_for_dim(nd)
    cls = [_clase(pr['name'], i) for i, pr in enumerate(probs)]
    print(f'\nD={nd}   mediana err(500k)/err(50k)   (menor = mejor aprovecha presupuesto)')
    print(f'  {"algoritmo":14s}' + ''.join(f'{c:>14s}' for c in CLASES))
    for a in ARMAS + COMPETIDORES:
        A, B = _load(a, nd, 50000), _load(a, nd, 500000)
        if A is None or B is None:
            continue
        ma, mb = A.mean(1), B.mean(1)
        row = []
        for c in CLASES:
            q = [i for i in range(71) if cls[i] == c and ma[i] > 0]
            row.append(np.median(mb[q] / ma[q]) if q else np.nan)
        print(f'  {a:14s}' + ''.join(f'{r:14.3f}' for r in row))
